# P8-3 · Bài 10b · Giám sát và chi phí

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q fastapi prometheus_client

## Lý thuyết

## Bài tập 5: Log và metric cho API {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json, time, uuid, logging
from fastapi import FastAPI, Request, Response
from prometheus_client import Histogram, Counter, generate_latest, CONTENT_TYPE_LATEST

app = FastAPI()
log = logging.getLogger("api"); logging.basicConfig(level=logging.INFO, format="%(message)s")
LATENCY = Histogram("request_latency_seconds", "Độ trễ request", ["path"], buckets=[0.1, 0.25, 0.5, 1, 2, 3, 5, 10])
ERRORS = Counter("request_errors_total", "Số request lỗi", ["path", "status"])

@app.middleware("http")
async def observe(request: Request, call_next):
    rid, t0 = str(uuid.uuid4()), time.perf_counter()
    response = await call_next(request)
    dt = time.perf_counter() - t0
    LATENCY.labels(request.url.path).observe(dt)
    if response.status_code >= 500:
        ERRORS.labels(request.url.path, str(response.status_code)).inc()
    log.info(json.dumps({"id": rid, "path": request.url.path, "status": response.status_code, "latency_s": round(dt, 3)}))
    response.headers["X-Request-ID"] = rid
    return response

@app.get("/metrics")
def metrics():
    return Response(generate_latest(), media_type=CONTENT_TYPE_LATEST)

# thử: gọi vài request rồi đọc metric như Prometheus sẽ đọc
from fastapi.testclient import TestClient
c = TestClient(app)
for _ in range(3): c.get("/metrics")
print([l for l in c.get("/metrics").text.splitlines() if l.startswith("request_latency_seconds_count")])

['request_latency_seconds_count{path="/metrics"} 3.0']


## Bài tập 6: Độ trễ: trung bình nói dối, phân vị nói thật {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
nhanh = rng.lognormal(np.log(0.6), 0.35, 9700)
cham = rng.lognormal(np.log(4.0), 0.4, 300)
lat = np.concatenate([nhanh, cham])               # giây
# TODO: tb, p50, p95, p99, vi_pham (tỉ lệ request > 2 giây)

In [ ]:
tb = lat.mean()
p50, p95, p99 = np.percentile(lat, [50, 95, 99])
vi_pham = (lat > 2).mean()
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.hist(lat, bins=np.logspace(np.log10(lat.min()), np.log10(lat.max()), 60))
ax.set_xscale("log")
for ten, v, c in [("p50", p50, "g"), ("p95", p95, "orange"), ("p99", p99, "r")]:
    ax.axvline(v, color=c, ls="--", label=f"{ten} = {v:.2f}s")
ax.legend(); ax.set(xlabel="độ trễ (giây, thang log)", ylabel="số request", title="Một nhóm nhỏ request rất chậm")
print(f"trung bình {tb:.2f}s | p50 {p50:.2f} | p95 {p95:.2f} | p99 {p99:.2f} | trên 2s: {vi_pham:.1%}")

trung bình 0.75s | p50 0.61 | p95 1.23 | p99 4.83 | trên 2s: 2.9%


## Bài tập 7: Phát hiện dữ liệu trôi bằng PSI {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
train = {"tuoi": rng.normal(35, 8, 20000), "thu_nhap": rng.lognormal(2.8, 0.5, 20000), "so_khoan_vay": rng.poisson(1.2, 20000)}
prod = {"tuoi": rng.normal(35.5, 8, 5000), "thu_nhap": rng.lognormal(3.1, 0.5, 5000), "so_khoan_vay": rng.poisson(1.3, 5000)}

def psi(a, b, bins=10):
    # TODO
    return 0.0

In [ ]:
def psi(a, b, bins=10):
    edges = np.unique(np.percentile(a, np.linspace(0, 100, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    p = np.histogram(a, edges)[0] / len(a) + 1e-6
    q = np.histogram(b, edges)[0] / len(b) + 1e-6
    return float(((q - p) * np.log(q / p)).sum())

for k in train:
    v = psi(train[k], prod[k])
    print(f"{k:13s} PSI = {v:.3f} {'← trôi đáng kể' if v > 0.2 else ''}")

tuoi          PSI = 0.009 
thu_nhap      PSI = 0.358 ← trôi đáng kể
so_khoan_vay  PSI = 0.007 


## Bài tập 8: A/B test: phiên bản mới có thật sự tốt hơn {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math
from scipy.stats import norm
xA, nA, xB, nB = 412, 2000, 468, 2000
# TODO: pA, pB, z, p_value; n_can (cỡ mẫu mỗi nhóm cho 20% → 22%)

In [ ]:
pA, pB = xA / nA, xB / nB
p = (xA + xB) / (nA + nB)
z = (pB - pA) / math.sqrt(p * (1 - p) * (1 / nA + 1 / nB))
p_value = 2 * (1 - norm.cdf(abs(z)))
p1, p2 = 0.20, 0.22; pbar = (p1 + p2) / 2
n_can = (norm.ppf(0.975) * math.sqrt(2 * pbar * (1 - pbar)) + norm.ppf(0.8) * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2 / (p2 - p1) ** 2
print(f"A {pA:.1%} → B {pB:.1%}, z = {z:.2f}, p-value = {p_value:.3f}")
print(f"cần khoảng {n_can:,.0f} người dùng mỗi nhóm để thấy được mức tăng 2 điểm phần trăm")

A 20.6% → B 23.4%, z = 2.14, p-value = 0.033
cần khoảng 6,509 người dùng mỗi nhóm để thấy được mức tăng 2 điểm phần trăm
